In [3]:
# ============================================================
# BiasTrace Adult RCA MI Ablation
# Compare label-agnostic MI I(Z;A) vs label-conditional MI I(Z;A|Y)
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr


# ------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------

ADULT_CSV_PATH = "/content/drive/MyDrive/Paper_Bias/exp/datasets/adult.data.csv"  # 필요하면 경로 수정

RANDOM_SEEDS = [0, 1, 2, 3, 4]
TOPK_VALUES = [1, 3, 5, 10]

PROTECTED_COL = "sex"
PROTECTED_POSITIVE_VALUE = "Male"

TARGET_COL = "income"
TARGET_POSITIVE_VALUE = ">50K"

# Adult에서 기존 BiasTrace repair 실험에 사용한 주요 carrier
MAIN_FEATURE = "hours-per-week"


# ------------------------------------------------------------
# 1. Adult loading and preprocessing
# ------------------------------------------------------------

ADULT_COLUMNS = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income",
]


def load_adult_raw(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Adult CSV not found: {path}\n"
            "Please upload adult.data.csv to Colab or update ADULT_CSV_PATH."
        )

    # Robust loading:
    # Some Adult files are headerless, while some include a header row.
    df_try = pd.read_csv(path, skipinitialspace=True)

    if TARGET_COL in df_try.columns and PROTECTED_COL in df_try.columns:
        df = df_try.copy()
    else:
        df = pd.read_csv(
            path,
            header=None,
            names=ADULT_COLUMNS,
            skipinitialspace=True,
            na_values=["?", " ?"]
        )

    # Strip whitespace in column names and object values
    df.columns = [str(c).strip() for c in df.columns]

    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()

    # Remove accidental header row if it was read as data
    # Example: row contains "age", "workclass", ..., "hours-per-week", "income"
    if len(df) > 0:
        header_like = False

        if "age" in df.columns and str(df.iloc[0]["age"]).strip().lower() == "age":
            header_like = True

        if TARGET_COL in df.columns and str(df.iloc[0][TARGET_COL]).strip().lower() == TARGET_COL:
            header_like = True

        if "hours-per-week" in df.columns and str(df.iloc[0]["hours-per-week"]).strip().lower() == "hours-per-week":
            header_like = True

        if header_like:
            df = df.iloc[1:].reset_index(drop=True)

    # Normalize income labels: >50K. -> >50K
    if TARGET_COL in df.columns:
        df[TARGET_COL] = (
            df[TARGET_COL]
            .astype(str)
            .str.replace(".", "", regex=False)
            .str.strip()
        )

    # Replace '?' strings with NaN
    df = df.replace("?", np.nan)

    # Convert known numeric columns
    numeric_adult_cols = [
        "age",
        "fnlwgt",
        "education-num",
        "capital-gain",
        "capital-loss",
        "hours-per-week",
    ]

    for c in numeric_adult_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=[PROTECTED_COL, TARGET_COL]).reset_index(drop=True)

    return df

def make_adult_base(df: pd.DataFrame):
    """
    Returns:
      base_df: raw feature dataframe for fault injection
      A: protected attribute, A=1 for Male, A=0 for Female
      y: target label, y=1 for >50K, y=0 for <=50K
    """
    A = (df[PROTECTED_COL] == PROTECTED_POSITIVE_VALUE).astype(int).to_numpy()
    y = (df[TARGET_COL] == TARGET_POSITIVE_VALUE).astype(int).to_numpy()

    drop_cols = [
        TARGET_COL,
        PROTECTED_COL,
    ]

    base_df = df.drop(columns=drop_cols, errors="ignore").copy()

    # Remove entirely missing columns
    base_df = base_df.dropna(axis=1, how="all")

    return base_df, A, y


# ------------------------------------------------------------
# 2. Fault injection scenarios for RCA MI ablation
# ------------------------------------------------------------

def inject_missingness(df, A, y, rng, feature=MAIN_FEATURE, rate=0.30):
    fdf = df.copy()
    mask = (A == 1) & (rng.random(len(fdf)) < rate)

    if feature in fdf.columns:
        fdf.loc[mask, feature] = np.nan

    return fdf, y.copy(), {"missing::" + feature, feature}


def inject_scale(df, A, y, rng, feature=MAIN_FEATURE, factor=1.50):
    fdf = df.copy()

    if feature in fdf.columns:
        values = pd.to_numeric(fdf[feature], errors="coerce")
        fdf.loc[A == 1, feature] = values.loc[A == 1] * factor

    return fdf, y.copy(), {feature}


def inject_shift(df, A, y, rng, feature=MAIN_FEATURE, shift=5.0):
    fdf = df.copy()

    if feature in fdf.columns:
        values = pd.to_numeric(fdf[feature], errors="coerce")
        fdf.loc[A == 1, feature] = values.loc[A == 1] + shift

    return fdf, y.copy(), {feature}


def inject_clip(df, A, y, rng, feature=MAIN_FEATURE, upper_quantile=0.70):
    fdf = df.copy()

    if feature in fdf.columns:
        values = pd.to_numeric(fdf[feature], errors="coerce")
        upper = values.quantile(upper_quantile)
        idx = (A == 1)
        fdf.loc[idx, feature] = np.minimum(values.loc[idx], upper)

    return fdf, y.copy(), {feature}

def inject_conditional_selection(df, A, y, rng, feature=MAIN_FEATURE, threshold=40, rate=0.35):
    """
    Remove a fraction of A=1 samples with hours-per-week > threshold.
    This is a distribution-shift fault, so multiple correlated carriers may change.
    """
    fdf = df.copy()

    cond = (A == 1)

    if feature in fdf.columns:
        feature_values = pd.to_numeric(fdf[feature], errors="coerce").fillna(-np.inf).to_numpy()
        cond = cond & (feature_values > threshold)

    remove = cond & (rng.random(len(fdf)) < rate)
    keep = ~remove

    return fdf.loc[keep].reset_index(drop=True), A[keep], y[keep], {feature}

def inject_missing_plus_scale(df, A, y, rng):
    fdf, y2, carriers1 = inject_missingness(
        df, A, y, rng, feature=MAIN_FEATURE, rate=0.20
    )
    fdf, y2, carriers2 = inject_scale(
        fdf, A, y2, rng, feature=MAIN_FEATURE, factor=1.25
    )
    return fdf, y2, carriers1.union(carriers2)


def inject_scale_plus_masking(df, A, y, rng):
    # RCA 관점에서는 upstream feature carrier만 평가한다.
    fdf, y2, carriers = inject_scale(
        df, A, y, rng, feature=MAIN_FEATURE, factor=1.50
    )
    return fdf, y2, carriers


SCENARIOS = {
    "S1_CONDITIONAL_SELECTION_HOURS": "conditional_selection",
    "S2_MISSING_HOURS": "missingness",
    "S3_SCALE_HOURS": "scale",
    "S3_SHIFT_HOURS": "shift",
    "S3_CLIP_HOURS": "clip",
    "M2_MISSING_PLUS_SCALE": "missing_plus_scale",
    "M3_SCALE_PLUS_MASKING": "scale_plus_masking",
}


def apply_scenario(base_df, A, y, scenario_name, seed):
    rng = np.random.default_rng(seed)
    kind = SCENARIOS[scenario_name]

    if kind == "conditional_selection":
        return inject_conditional_selection(base_df, A, y, rng)

    if kind == "missingness":
        fdf, y2, carriers = inject_missingness(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    if kind == "scale":
        fdf, y2, carriers = inject_scale(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    if kind == "shift":
        fdf, y2, carriers = inject_shift(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    if kind == "clip":
        fdf, y2, carriers = inject_clip(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    if kind == "missing_plus_scale":
        fdf, y2, carriers = inject_missing_plus_scale(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    if kind == "scale_plus_masking":
        fdf, y2, carriers = inject_scale_plus_masking(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    raise ValueError(f"Unknown scenario kind: {kind}")


# ------------------------------------------------------------
# 3. Carrier matrix construction
# ------------------------------------------------------------

def build_carrier_matrix(raw_df: pd.DataFrame):
    """
    Construct RCA carrier matrix.
    Includes:
      - missingness indicators
      - numeric features
      - one-hot categorical features
    """
    df = raw_df.copy()

    carrier_parts = []
    feature_names = []
    discrete_mask = []

    # Missingness indicators before imputation
    for c in df.columns:
        if df[c].isna().any():
            miss = df[c].isna().astype(int).to_numpy().reshape(-1, 1)
            carrier_parts.append(miss)
            feature_names.append(f"missing::{c}")
            discrete_mask.append(True)

    # Separate numeric and categorical
    numeric_cols = df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [c for c in df.columns if c not in numeric_cols]

    # Numeric imputation and scaling
    if numeric_cols:
        num_df = df[numeric_cols].copy()

        for c in numeric_cols:
            med = num_df[c].median()
            if pd.isna(med):
                med = 0.0
            num_df[c] = num_df[c].fillna(med)

        scaler = StandardScaler()
        X_num = scaler.fit_transform(num_df.to_numpy(dtype=float))

        carrier_parts.append(X_num)
        feature_names.extend(numeric_cols)
        discrete_mask.extend([False] * len(numeric_cols))

    # Categorical imputation and one-hot
    if categorical_cols:
        cat_df = df[categorical_cols].copy()

        for c in categorical_cols:
            mode = cat_df[c].mode(dropna=True)
            fill_value = mode.iloc[0] if len(mode) > 0 else "MISSING"
            cat_df[c] = cat_df[c].fillna(fill_value).astype(str)

        dummies = pd.get_dummies(cat_df, drop_first=False)
        X_cat = dummies.to_numpy(dtype=float)

        carrier_parts.append(X_cat)
        feature_names.extend(dummies.columns.tolist())
        discrete_mask.extend([True] * X_cat.shape[1])

    if not carrier_parts:
        raise ValueError("No usable carrier features found.")

    X = np.hstack(carrier_parts)
    discrete_mask = np.array(discrete_mask, dtype=bool)

    return X, feature_names, discrete_mask


# ------------------------------------------------------------
# 4. MI and conditional MI scoring
# ------------------------------------------------------------

def safe_mutual_info(X, A, discrete_mask, seed=0):
    A = np.asarray(A).astype(int)

    if len(np.unique(A)) < 2:
        return np.zeros(X.shape[1])

    try:
        scores = mutual_info_classif(
            X,
            A,
            discrete_features=discrete_mask,
            random_state=seed,
            n_neighbors=3
        )
    except Exception as e:
        print(f"[WARN] mutual_info_classif failed: {e}")
        scores = np.zeros(X.shape[1])

    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    return scores


def label_agnostic_mi(X, A, y, discrete_mask, seed=0):
    return safe_mutual_info(X, A, discrete_mask, seed=seed)


def label_conditional_mi(X, A, y, discrete_mask, seed=0):
    """
    Approximate I(Z;A|Y) = sum_y P(Y=y) I(Z;A | Y=y).
    """
    y = np.asarray(y).astype(int)
    A = np.asarray(A).astype(int)

    scores = np.zeros(X.shape[1], dtype=float)

    for yy in np.unique(y):
        idx = (y == yy)
        weight = idx.mean()

        if idx.sum() < 20:
            continue
        if len(np.unique(A[idx])) < 2:
            continue

        scores_y = safe_mutual_info(
            X[idx],
            A[idx],
            discrete_mask,
            seed=seed
        )
        scores += weight * scores_y

    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    return scores


# ------------------------------------------------------------
# 5. Ranking and evaluation
# ------------------------------------------------------------

def normalize_feature_name(name: str) -> str:
    return str(name).replace(" ", "_").lower()


def is_carrier_match(feature_name: str, gt_carriers: set):
    f = normalize_feature_name(feature_name)

    for gt in gt_carriers:
        g = normalize_feature_name(gt)

        if f == g:
            return True

        # One-hot prefix match
        if f.startswith(g + "_"):
            return True

        if g.startswith("missing::") and f == g:
            return True

    return False


def rank_features(scores, feature_names):
    order = np.argsort(-scores)

    ranked = []
    for rank, idx in enumerate(order, start=1):
        ranked.append({
            "rank": rank,
            "feature": feature_names[idx],
            "score": float(scores[idx])
        })

    return ranked


def evaluate_ranked(ranked, gt_carriers, topk_values=TOPK_VALUES):
    ranks = [
        row["rank"]
        for row in ranked
        if is_carrier_match(row["feature"], gt_carriers)
    ]

    best_rank = min(ranks) if ranks else np.inf

    out = {
        "best_rank": best_rank if np.isfinite(best_rank) else None
    }

    for k in topk_values:
        out[f"hit@{k}"] = int(np.isfinite(best_rank) and best_rank <= k)

    return out


def spearman_between_scores(scores_a, scores_b):
    if np.all(scores_a == scores_a[0]) or np.all(scores_b == scores_b[0]):
        return np.nan

    corr, _ = spearmanr(scores_a, scores_b)
    return float(corr)


# ------------------------------------------------------------
# 6. Main experiment
# ------------------------------------------------------------

def run_adult_mi_ablation(
    adult_csv_path=ADULT_CSV_PATH,
    scenarios=None,
    seeds=RANDOM_SEEDS,
    save_prefix="biastrace_adult_mi_ablation"
):
    if scenarios is None:
        scenarios = list(SCENARIOS.keys())

    raw = load_adult_raw(adult_csv_path)
    base_df, A, y = make_adult_base(raw)

    print("========== ADULT DATA INFO ==========")
    print("Rows:", len(base_df))
    print("Protected A count [0,1]:", np.bincount(A))
    print("Target y count [0,1]:", np.bincount(y))
    print("Base feature columns:", len(base_df.columns))
    print(base_df.columns.tolist())

    rows = []
    top_rows = []

    for scenario in scenarios:
        for seed in seeds:
            fdf, Af, yf, gt_carriers = apply_scenario(base_df, A, y, scenario, seed)

            X, feature_names, discrete_mask = build_carrier_matrix(fdf)

            scores_mi = label_agnostic_mi(X, Af, yf, discrete_mask, seed=seed)
            scores_cmi = label_conditional_mi(X, Af, yf, discrete_mask, seed=seed)

            ranked_mi = rank_features(scores_mi, feature_names)
            ranked_cmi = rank_features(scores_cmi, feature_names)

            eval_mi = evaluate_ranked(ranked_mi, gt_carriers)
            eval_cmi = evaluate_ranked(ranked_cmi, gt_carriers)

            corr = spearman_between_scores(scores_mi, scores_cmi)

            for mi_type, ranked, ev in [
                ("I(Z;A)", ranked_mi, eval_mi),
                ("I(Z;A|Y)", ranked_cmi, eval_cmi),
            ]:
                row = {
                    "dataset": "Adult",
                    "scenario": scenario,
                    "seed": seed,
                    "mi_type": mi_type,
                    "gt_carriers": ";".join(sorted(gt_carriers)),
                    "best_rank": ev["best_rank"],
                    "rank_corr_mi_vs_cmi": corr,
                }

                for k in TOPK_VALUES:
                    row[f"hit@{k}"] = ev[f"hit@{k}"]

                rows.append(row)

                for r in ranked[:10]:
                    top_rows.append({
                        "dataset": "Adult",
                        "scenario": scenario,
                        "seed": seed,
                        "mi_type": mi_type,
                        "rank": r["rank"],
                        "feature": r["feature"],
                        "score": r["score"],
                        "gt_carriers": ";".join(sorted(gt_carriers)),
                        "is_gt_match": int(is_carrier_match(r["feature"], gt_carriers))
                    })

    result_df = pd.DataFrame(rows)
    top_df = pd.DataFrame(top_rows)

    summary = (
        result_df
        .groupby(["dataset", "mi_type"])
        .agg(
            n=("scenario", "count"),
            rca_r1=("hit@1", "mean"),
            rca_r3=("hit@3", "mean"),
            rca_r5=("hit@5", "mean"),
            rca_r10=("hit@10", "mean"),
            mean_best_rank=("best_rank", "mean"),
            mean_rank_corr=("rank_corr_mi_vs_cmi", "mean"),
        )
        .reset_index()
    )

    scenario_summary = (
        result_df
        .groupby(["scenario", "mi_type"])
        .agg(
            n=("seed", "count"),
            rca_r1=("hit@1", "mean"),
            rca_r3=("hit@3", "mean"),
            rca_r5=("hit@5", "mean"),
            mean_best_rank=("best_rank", "mean"),
            mean_rank_corr=("rank_corr_mi_vs_cmi", "mean"),
        )
        .reset_index()
    )

    for df in [summary, scenario_summary]:
        for col in ["rca_r1", "rca_r3", "rca_r5", "rca_r10"]:
            if col in df.columns:
                df[col] = df[col] * 100.0

    result_path = f"{save_prefix}_per_run.csv"
    top_path = f"{save_prefix}_top10.csv"
    summary_path = f"{save_prefix}_summary.csv"
    scenario_summary_path = f"{save_prefix}_scenario_summary.csv"

    result_df.to_csv(result_path, index=False)
    top_df.to_csv(top_path, index=False)
    summary.to_csv(summary_path, index=False)
    scenario_summary.to_csv(scenario_summary_path, index=False)

    print("\n========== SUMMARY ==========")
    print(summary.to_string(index=False))

    print("\n========== SCENARIO SUMMARY ==========")
    print(scenario_summary.to_string(index=False))

    print("\nSaved files:")
    print(" -", result_path)
    print(" -", top_path)
    print(" -", summary_path)
    print(" -", scenario_summary_path)

    return result_df, top_df, summary, scenario_summary


# ------------------------------------------------------------
# 7. Run
# ------------------------------------------------------------

result_df, top_df, summary_df, scenario_summary_df = run_adult_mi_ablation()

========== ADULT DATA INFO ==========
Rows: 48842
Protected A count [0,1]: [16192 32650]
Target y count [0,1]: [37155 11687]
Base feature columns: 13
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']


/tmp/ipykernel_7216/42306615.py:175: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[60.  19.5 60.  ... 75.  60.  90. ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fdf.loc[A == 1, feature] = values.loc[A == 1] * factor
/tmp/ipykernel_7216/42306615.py:175: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[60.  19.5 60.  ... 75.  60.  90. ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fdf.loc[A == 1, feature] = values.loc[A == 1] * factor
/tmp/ipykernel_7216/42306615.py:175: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[60.  19.5 60.  ... 75.  60.  90. ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fdf.loc[A == 1


========== SUMMARY ==========
dataset  mi_type  n    rca_r1    rca_r3    rca_r5  rca_r10  mean_best_rank  mean_rank_corr
  Adult   I(Z;A) 35 57.142857 71.428571 85.714286    100.0        2.800000        0.924416
  Adult I(Z;A|Y) 35 57.142857 85.714286 85.714286    100.0        2.685714        0.924416

========== SCENARIO SUMMARY ==========
                      scenario  mi_type  n  rca_r1  rca_r3  rca_r5  mean_best_rank  mean_rank_corr
         M2_MISSING_PLUS_SCALE   I(Z;A)  5   100.0   100.0   100.0             1.0        0.926610
         M2_MISSING_PLUS_SCALE I(Z;A|Y)  5   100.0   100.0   100.0             1.0        0.926610
         M3_SCALE_PLUS_MASKING   I(Z;A)  5   100.0   100.0   100.0             1.0        0.924524
         M3_SCALE_PLUS_MASKING I(Z;A|Y)  5   100.0   100.0   100.0             1.0        0.924524
S1_CONDITIONAL_SELECTION_HOURS   I(Z;A)  5     0.0     0.0     0.0             8.6        0.919597
S1_CONDITIONAL_SELECTION_HOURS I(Z;A|Y)  5     0.0     0.0    